# Exploratory Data Analysis — IEEE-CIS Fraud Detection

IEEE-CIS Fraud Detection veri setinin ilk keşifsel analizi: veri boyutu, eksik değer yapısı, hedef değişken (`isFraud`) dağılımı, işlem tutarı ve zaman aralığı.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)

## 1. Veri Yükleme ve Birleştirme

`transaction` ve `identity` tabloları `TransactionID` üzerinden left join ile birleştirilir; her işlemin bir identity (cihaz/tarayıcı) kaydı bulunmayabilir.

In [ ]:
df_transaction = pd.read_csv("../data/train_transaction.csv")
df_identity = pd.read_csv("../data/train_identity.csv")
df = pd.merge(df_transaction, df_identity, on="TransactionID", how="left")

## 2. Genel Bakış

In [ ]:
print(df.shape)

In [ ]:
df.info()

In [ ]:
df.head()

## 3. Eksik Değer Analizi

In [ ]:
missing_pct = df.isna().mean()
missing_pct.sort_values(ascending=False).head(20)

Eksik değer oranları dört aralıkta gruplandırılmıştır:

In [ ]:
missing_buckets = pd.cut(
    missing_pct,
    bins=[0.0, 0.1, 0.5, 0.9, 1.0],
    labels=["%0-10", "%10-50", "%50-90", "%90-100"],
    include_lowest=True,
)
missing_buckets.value_counts()

**Bulgu:** 434 sütunun ~%47'si (202 sütun) `%50-90` aralığında eksik değere sahip; yalnızca 12 sütun `%90`'ın üzerinde eksik. Bu dağılım, feature engineering aşamasında hangi ham sütunların doğrudan kullanılacağına, hangilerinin bir "eksik mi" bayrağıyla temsil edileceğine karar verirken referans alınacaktır.

## 4. Hedef Değişken (`isFraud`) Dağılımı

In [ ]:
fraud_class = df["isFraud"].value_counts(normalize=True)
fraud_class

In [ ]:
sns.countplot(data=df, x="isFraud")

**Bulgu:** İşlemlerin %96.5'i normal, %3.5'i dolandırıcılık — ciddi bir sınıf dengesizliği söz konusu. Bu nedenle model değerlendirmesinde accuracy yerine PR-AUC esas alınacak; sınıf dengesizliği SMOTE ile yapay örnekleme yerine class weight (maliyet-duyarlı öğrenme) ile ele alınacaktır.

## 5. TransactionAmt ve Zaman (TransactionDT) Analizi

İşlem tutarının (`TransactionAmt`) fraud ve normal işlemler arasındaki dağılımı ile veri setinin kapsadığı zaman aralığı (`TransactionDT`) incelenmiştir.

In [ ]:
df.groupby("isFraud")["TransactionAmt"].describe()

In [ ]:
sns.boxplot(data=df, x="isFraud", y="TransactionAmt")
plt.yscale("log")

**Bulgu:** Fraud işlemlerin ortalama (149.2) ve medyan (75.0) tutarları normal işlemlerden (sırasıyla 134.5 ve 68.5) biraz daha yüksek; ancak asıl belirgin fark dağılımın genişliğinde — fraud işlemlerin orta %50'lik aralığı (IQR ≈ 126) normal işlemlerin neredeyse iki katı (IQR ≈ 76), yani fraud işlemler tutar açısından daha öngörülemez. İki dağılım büyük ölçüde çakıştığından ham `TransactionAmt` tek başına güçlü bir ayırt edici değil; kullanıcının kendi harcama geçmişinden sapmayı ölçen türetilmiş özellikler (`amount_deviation_from_user`) daha değerli olacaktır.

In [ ]:
transaction_dt_min_days = df["TransactionDT"].min() / (60 * 60 * 24)
transaction_dt_max_days = df["TransactionDT"].max() / (60 * 60 * 24)
print(transaction_dt_min_days, transaction_dt_max_days)

**Bulgu:** `TransactionDT`, gerçek bir takvim tarihi değil, bir referans noktasından itibaren geçen saniye sayısıdır. Veri seti yaklaşık 183 günlük (~6 aylık) bir dönemi kapsamaktadır. Bu, model geliştirme aşamasında uygulanacak zaman bazlı (temporal) train/test ayrımı için veri setinin toplam süresini netleştirir — örneğin ilk ~5 ay train, son ~1 ay test olarak kullanılabilir; rastgele bölme, gelecekteki bilgiyi geçmişi tahmin etmekte kullanma riski (zaman sızıntısı) taşır.

## 6. Kategorik Değişkenler → Fraud Rate

Kategorik sütunlarda fraud oranı hesaplanırken işlem sayısı (`count`) da dikkate alınmalıdır — az örnekli kategorilerde fraud oranı istatistiksel olarak güvenilir olmayabilir. Bu analiz için tekrar kullanılabilir bir fonksiyon tanımlanmıştır.

In [ ]:
def fraud_rate_by_category(df, column):
    result = df.groupby(column)["isFraud"].agg(["count", "mean"])
    return result.sort_values("mean", ascending=False)

In [ ]:
fraud_rate_by_category(df, "ProductCD")

In [ ]:
fraud_rate_by_category(df, "card4")

In [ ]:
fraud_rate_by_category(df, "card6")

In [ ]:
fraud_rate_by_category(df, "DeviceType")

**Bulgu:** Kategorik sütunlar, `TransactionAmt`'den çok daha güçlü ayırt edici sinyaller taşıyor:
- `ProductCD`: `C` kategorisi (%11.7 fraud) `W`'ye (%2.0, işlemlerin ~%74'ü) göre ~6 kat daha riskli.
- `card4`: `discover` en yüksek oranlı (%7.7) ama en küçük örneklemli (6,651); `visa`/`mastercard` birbirine yakın (~%3.5).
- `card6`: `credit` (%6.7) `debit`'e (%2.4) göre ~2.8 kat daha riskli; `charge card`/`debit or credit` kategorileri örneklem büyüklüğü çok küçük olduğundan (15-30 işlem) güvenilmez.
- `DeviceType`: yalnızca identity kaydı olan işlemlerde mevcut; `mobile` (%10.2) `desktop`'a (%6.5) göre daha riskli — ikisi de genel ortalamanın (%3.5) belirgin şekilde üzerinde, bu da identity verisi olan işlemlerin genel olarak daha riskli olabileceğine işaret ediyor (7. bölümde inceleniyor).

## 7. Missingness → Fraud Rate

**Görev:** İşlemin bir identity (cihaz/tarayıcı) kaydına sahip olup olmadığını gösteren yeni bir sütun oluştur, sonra `fraud_rate_by_category` fonksiyonunu (6. bölümde yazdığın) bu sütuna uygula.

**Adımlar:**
1. `id_01` sütunu, identity tablosunda hiç eksik değeri olmayan bir sütundu (hatırlarsan `df_identity.info()` çıktısında 144233/144233 doluydu). Birleştirilmiş `df`'te bu sütun `NaN` ise, o işlemin identity kaydı **yok** demektir.
2. `has_identity` adında, `id_01`'in dolu olup olmadığını gösteren `True`/`False` bir sütun oluştur.
3. `fraud_rate_by_category(df, "has_identity")` çağır.

**İpucu:** `.notna()` bir sütunun dolu (eksik olmayan) hücrelerini `True` olarak işaretler.

In [ ]:
df["has_identity"] = df["id_01"].notna()
fraud_rate_by_category(df, "has_identity")

**Bulgu:** Identity (cihaz/tarayıcı) kaydı olan işlemler (%7.85 fraud, n=144,233) olmayanlara (%2.09 fraud, n=446,307) göre ~3.75 kat daha riskli. Bu, identity verisinin muhtemelen belirli, doğası gereği daha yüksek riskli kanallardan (örn. card-not-present) toplandığını düşündürüyor. Daha genel bir çıkarım: bir sütunun **eksik olması bile başlı başına bir sinyal** taşıyabilir — feature engineering aşamasında ham `id_*` sütunlarının yanı sıra bu tür "eksik mi" bayraklarının da modele girdi olarak verilmesi değerlidir.

## 8. Korelasyon / Feature Importance (Sayısal Sütunlar)

370'i aşkın anonimleştirilmiş sayısal sütunu (`V1-V339`, `C1-C14`, `D1-D15` vb.) tek tek görselleştirmek yerine, hepsinin `isFraud` ile **korelasyonunu** hesaplayıp en yüksek ilişkili olanları önceliklendireceğiz.

**Görev:**
1. `df`'ten sayısal sütunları seç (`select_dtypes`), ama `isFraud` (hedefin kendisi), `TransactionID` (kimlik, anlamsız) ve `TransactionDT`'yi (zamanı zaten ayrı inceliyoruz) hariç tut.
2. Her sayısal sütun ile `isFraud` arasındaki korelasyonu hesapla.
3. Mutlak değere göre büyükten küçüğe sırala, ilk 15 sütunu göster.

**İpucu:** `df.select_dtypes(include=np.number)`, bir DataFrame'in her sütununu tek bir Series ile karşılaştırmak için `.corrwith(seri)` (döngü yazmana gerek yok, pandas bunu senin için yapar), mutlak değer için `.abs()`.

**Dikkat:** Korelasyon yalnızca **doğrusal** ilişkileri yakalar ve eksik değeri çok olan sütunlarda (bkz. 3. bölüm) az sayıda veriden hesaplanacağı için güvenilirliği düşük olabilir — bu, kesin bir feature importance değil, sadece bir ön tarama.

In [ ]:
numeric_df = df.select_dtypes(include=np.number).drop(columns=["isFraud", "TransactionID", "TransactionDT"])
corr_with_fraud = numeric_df.corrwith(df["isFraud"])

In [ ]:
top_corr = corr_with_fraud.abs().sort_values(ascending=False).head(15)
top_corr

In [ ]:
missing_pct[top_corr.index]

**Bulgu:** `isFraud` ile en yüksek korelasyona sahip 15 sütunun tamamı anonimleştirilmiş `V*` sütunları (`0.26`–`0.38` arası) — `card1-6`, `addr1/2`, `dist1/2`, `TransactionAmt` gibi yorumlanabilir sütunların hiçbiri bu güce ulaşmıyor (en güçlüsü `card3` ~0.15). Bu sütunların çoğu `%76-86` oranında eksik; mutlak örneklem büyüklüğü hâlâ yeterli (~80-130 bin satır) olduğundan küçük-örneklem riski yok, ancak eksikliğin kendisinin fraud ile ilişkili olduğunu (7. bölüm) göz önünde bulundurunca, bu korelasyonların bir kısmı "hangi alt grupta bu alan dolu" bilgisini de yansıtıyor olabilir. Sonuç olarak bu 15 sütun feature engineering için güçlü adaylar, ancak kesin önem sırası ağaç bazlı modellerin (XGBoost/LightGBM) eğitimi sonrası netleşecektir.

## 9. Zaman (TransactionDT) → Fraud Rate

Veri setinin ~183 günlük döneminde fraud oranının zamanla nasıl değiştiğini (sabit mi, artıyor/azalıyor mu, ani sıçramalar var mı) inceleyeceğiz — bu, hem temporal split kararımızı hem de modelin zamanla "eskiyip eskimeyeceği" (concept drift) sorusunu ilgilendiriyor.

**Görev:**
1. `TransactionDT`'yi (saniye) gün indeksine çevir: `df["transaction_day"] = df["TransactionDT"] // (60 * 60 * 24)`. `//` operatörü **tam sayı bölmesi** yapar (kalanı atar) — örneğin `86400 // 86400 = 1`, `100000 // 86400 = 1`, `172800 // 86400 = 2`. Böylece her işlem 0-182 arası bir gün numarasına düşer.
2. `transaction_day`'e göre grupla, her günün fraud oranını (`isFraud` ortalaması) hesapla.
3. Sonucu bir çizgi grafikle (`sns.lineplot` veya `.plot()`) günlere göre görselleştir.

**İpucu:** `df.groupby("transaction_day")["isFraud"].mean()` bir Series döndürür, Series'lerin de `.plot()` metodu vardır (index otomatik olarak x ekseni olur).

In [ ]:
df["transaction_day"] = df["TransactionDT"] // (60 * 60 * 24)
daily_fraud_rate = df.groupby("transaction_day")["isFraud"].mean()
daily_fraud_rate.plot()

**Bulgu:** 7 günlük hareketli ortalama gürültüyü temizleyip net bir örüntü ortaya çıkardı: ilk ~25 gün fraud oranı daha düşük (~%2-3), sonrasında %3.5-5 bandına yükselip geri kalan ~155 günde bu aralıkta kalıyor. Asıl "farklı" olan dönem veri setinin başlangıcı; test için ayıracağımız son ay, eğitim döneminin geri kalanıyla benzer bir rejimde — bu, temporal split kararımız için riski azaltan bir bulgu.

## 10. Veri Kalitesi Kontrolleri

### 10a. Tam Kopya Satırlar

**Görev:** `df`'te tamamen birbirinin aynı olan (tüm sütunlarda aynı değerlere sahip) satır var mı kontrol et.

**İpucu:** `.duplicated()` her satır için "bu satır daha önce görülmüş mü?" diye `True`/`False` döndürür, `.sum()` ile kaç tane olduğunu sayabilirsin (`True` = 1, `False` = 0 olarak toplanır).

### 10b. Neredeyse Sabit Sütunlar

**Görev:** Her sütunun en sık tekrar eden değerinin, o sütunun **yüzde kaçını** oluşturduğunu hesapla — eğer bir sütunun %99'undan fazlası tek bir değerse (örn. hep aynı sayı), o sütun modele neredeyse hiç bilgi katmaz.

**Adımlar:**
1. Boş bir sözlük (`{}`) oluştur.
2. `df.columns` üzerinde `for` döngüsüyle her sütunu gez.
3. Her sütun için `df[col].value_counts(normalize=True, dropna=False).iloc[0]` — bu, o sütundaki **en sık görülen değerin oranını** verir (`value_counts` varsayılan olarak azalan sırada döndürür, `.iloc[0]` ilk/en yüksek satırı alır; `dropna=False` eksik değerleri de bir "kategori" olarak sayar).
4. Sözlüğü bir Series'e çevir (`pd.Series(sözlük)`), `%99`'un üzerindeki sütunları filtrele.

**İpucu (döngü + sözlük deseni):**
```python
top_value_ratio = {}
for col in df.columns:
    top_value_ratio[col] = ...
top_value_ratio = pd.Series(top_value_ratio)
```

In [ ]:
duplicate_rows = df.duplicated().sum()
duplicate_rows

In [ ]:
top_value_ratio = {}
for col in df.columns:
    top_value_ratio[col] = df[col].value_counts(normalize=True, dropna=False).iloc[0]
top_value_ratio = pd.Series(top_value_ratio)
top_value_ratio[top_value_ratio > 0.99].sort_values(ascending=False)

**Bulgu:** Tam kopya satır yok (veri kalitesi bu açıdan temiz). En sık değer oranı `%99`'u aşan 23 sütun iki farklı grupta toplanıyor: (1) `id_07/08/21-27` — bunlar 3. bölümde tespit edilen, zaten `%99+` eksik olan sütunlar (baskın "değer" aslında `NaN`); (2) `V107-V122` aralığı, `V305`, `C3` — bunlar çoğunlukla dolu ama gerçek değerleri neredeyse hep aynı, yani gerçek anlamda düşük varyanslı sütunlar. İkinci grup, feature engineering öncesi elenmeye aday (modele ayırt edici bilgi katmıyor); birinci grup zaten missingness analizinde ele alınacak.

## 11. EDA Özeti

**Veri:** 590,540 işlem, 434 sütun (transaction + identity, `TransactionID` üzerinden left join). Identity kaydı işlemlerin yalnızca ~%24'ünde mevcut.

**Eksik değerler:** Sütunların ~%47'si `%50-90` aralığında eksik; 12 sütun `%90`'ın üzerinde. Eksiklik rastgele değil — identity verisi olmayan işlemler (`has_identity=False`) genel ortalamadan (%3.5) daha düşük fraud oranına (%2.09) sahipken, identity verisi olanlar (%7.85) belirgin şekilde daha risklidir. Bu nedenle eksiklik, ham değerlerin yanı sıra "eksik mi" bayrakları olarak da feature engineering'e taşınmalıdır.

**Hedef değişken:** Ciddi sınıf dengesizliği — %96.5 normal, %3.5 fraud. Model değerlendirmesinde accuracy yerine PR-AUC, dengesizlik için SMOTE yerine class weight kullanılacaktır.

**Sayısal/kategorik sinyaller:**
- `TransactionAmt` tek başına zayıf bir ayırt edici (dağılımlar büyük ölçüde çakışıyor), ancak fraud işlemlerde IQR daha geniş.
- Kategorik sütunlar güçlü sinyaller taşıyor: `ProductCD` (`C` ~6 kat `W`'den riskli), `card6` (`credit` ~2.8 kat `debit`'ten riskli), `DeviceType` (`mobile` > `desktop`).
- `isFraud` ile en yüksek korelasyona sahip 15 sütunun tamamı anonimleştirilmiş `V*` sütunları (0.26–0.38) — ham kart/adres bilgisinden daha güçlü, ancak bir kısmı yüksek eksiklik nedeniyle dikkatli yorumlanmalı.

**Zaman:** Veri seti ~183 günü kapsıyor. Fraud oranı ilk ~25 günde daha düşük, sonrasında %3.5-5 bandında kararlı — temporal split için ciddi bir rejim uyumsuzluğu riski görünmüyor.

**Veri kalitesi:** Tam kopya satır yok. 23 sütun `%99+` oranında tek bir değer taşıyor; bir kısmı zaten bilinen yüksek-eksiklik sütunları, bir kısmı (`V107-V122`, `V305`, `C3`) gerçek anlamda düşük varyanslı — bu ikinci grup feature engineering öncesi elenmeye adaydır.

**Sonraki adım:** Bu bulgular ışığında feature engineering aşamasına geçilecek — proje kapsamındaki türetilmiş özellikler (`transactions_last_10min`, `amount_deviation_from_user`, `new_device`, `new_location` vb.) bu EDA'da tespit edilen örüntüler (kullanıcı bazlı sapmenin ham tutardan daha değerli olması, eksikliğin sinyal taşıması, kategorik risk farklılıkları) referans alınarak kodlanacaktır.